**Jakub Orchowski, s223281**

# CEL ĆWICZENIA
2D transformacje geometryczne obrazów

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
%matplotlib inline

# Zadania

## Zadanie 1.
a) Napisz funkcję do przekształcenia współrzędnych matematycznych $(x_i, y_i)$ we współrzędne obrazu cyfrowego o rozdzielczości $m \times n$ (tzn. $(x_{PIX}, y_{PIX})$). Załóż rozmiar pixela $\Delta x \times \Delta y$.

b) Napisz funkcję dokonującą transformacji odwrotnej.

Rozwiąż oba problemy używając formalizmu współrzędnych jednorodnych.

### (a) Współrzędne matematyczne → pikselowe

In [ ]:
def math_to_pixel(x, y, m, n, dx, dy):
    """Przekształcenie współrzędnych matematycznych (x, y) na pikselowe (xPIX, yPIX).
    Obraz ma rozdzielczość m (wiersze) x n (kolumny), rozmiar pixela dx x dy.

    Układ matematyczny: początek w środku obrazu, oś Y w górę.
    Układ pikselowy: początek w lewym górnym rogu, oś Y w dół.

    Macierz transformacji (współrzędne jednorodne):
    [xPIX]   [1/dx    0    n/2] [x]
    [yPIX] = [ 0    -1/dy  m/2] [y]
    [  1 ]   [ 0      0     1 ] [1]
    """
    H = np.array([
        [1.0/dx,    0.0,   n/2.0],
        [  0.0,  -1.0/dy,  m/2.0],
        [  0.0,     0.0,    1.0 ]
    ])

    coords = np.array([x, y, np.ones_like(x)])
    result = H @ coords
    return result[0], result[1], H

### (b) Współrzędne pikselowe → matematyczne

In [ ]:
def pixel_to_math(xPIX, yPIX, m, n, dx, dy):
    """Przekształcenie współrzędnych pikselowych (xPIX, yPIX) na matematyczne (x, y).
    Transformacja odwrotna do math_to_pixel.
    """
    H = np.array([
        [1.0/dx,    0.0,   n/2.0],
        [  0.0,  -1.0/dy,  m/2.0],
        [  0.0,     0.0,    1.0 ]
    ])
    H_inv = np.linalg.inv(H)

    coords = np.array([xPIX, yPIX, np.ones_like(xPIX)])
    result = H_inv @ coords
    return result[0], result[1], H_inv

### Test obu funkcji

In [ ]:
test_cases = [
    {'m': 100, 'n': 200, 'dx': 1.0, 'dy': 1.0, 'label': '100x200, dx=dy=1'},
    {'m': 480, 'n': 640, 'dx': 0.5, 'dy': 0.5, 'label': '480x640, dx=dy=0.5'},
    {'m': 300, 'n': 400, 'dx': 2.0, 'dy': 1.5, 'label': '300x400, dx=2, dy=1.5'},
]

x_test = np.array([0.0, 10.0, -10.0, 5.0, -5.0])
y_test = np.array([0.0, 10.0, -10.0, -5.0, 5.0])

for tc in test_cases:
    m, n, dx, dy = tc['m'], tc['n'], tc['dx'], tc['dy']
    xp, yp, H = math_to_pixel(x_test, y_test, m, n, dx, dy)
    x_back, y_back, H_inv = pixel_to_math(xp, yp, m, n, dx, dy)

    err = np.max(np.abs(x_test - x_back) + np.abs(y_test - y_back))
    print(f'{tc["label"]}: max blad round-trip = {err:.2e}')
    print(f'  Przykladowy punkt: ({x_test[1]}, {y_test[1]}) -> pix ({xp[1]:.1f}, {yp[1]:.1f}) -> math ({x_back[1]:.4f}, {y_back[1]:.4f})')
    print()

### Wnioski
Transformacja math→pixel i jej odwrotność pixel→math dają zerowy błąd round-trip.

Formalizm współrzędnych jednorodnych pozwala wyrazić obie transformacje jako mnożenie macierzy 3×3,
co upraszcza składanie transformacji.

## Zadanie 2.
Parametry transformacji pomiędzy dwoma obrazami wyznaczane przez porównanie
współrzędnych odpowiadających sobie pikseli.

a) Przekształcenie podobieństwa (4 parametry → minimum 2 pary punktów)

b) Przekształcenie afiniczne (6 parametrów → minimum 3 pary punktów)

Dane: A.bmp, B.bmp oraz A_podo.bmp, B_podo.bmp, A_afin.bmp, B_afin.bmp

In [ ]:
Im_A = np.array(Image.open('A.bmp'))
Im_B = np.array(Image.open('B.bmp'))
Im_A_podo = np.array(Image.open('A_podo.bmp'))
Im_B_podo = np.array(Image.open('B_podo.bmp'))
Im_A_afin = np.array(Image.open('A_afin.bmp'))
Im_B_afin = np.array(Image.open('B_afin.bmp'))

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, img, title in zip(axes[0], [Im_A, Im_A_podo, Im_A_afin],
                           ['A.bmp', 'A_podo.bmp', 'A_afin.bmp']):
    ax.imshow(img, cmap='gray')
    ax.set_title(title)
    ax.axis('off')
for ax, img, title in zip(axes[1], [Im_B, Im_B_podo, Im_B_afin],
                           ['B.bmp', 'B_podo.bmp', 'B_afin.bmp']):
    ax.imshow(img, cmap='gray')
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()

### (a) Wyznaczanie macierzy podobieństwa

In [ ]:
def find_similarity(src_pts, dst_pts):
    """Wyznaczanie macierzy przekształcenia podobieństwa na podstawie par punktów.

    Podobieństwo ma postać:
    [x']   [a  -b  tx] [x]
    [y'] = [b   a  ty] [y]
    [ 1]   [0   0   1] [1]

    4 niewiadome (a, b, tx, ty) → minimum 2 pary punktów.
    """
    n = src_pts.shape[0]
    A = np.zeros((2 * n, 4))
    b = np.zeros(2 * n)

    for i in range(n):
        x, y = src_pts[i]
        xp, yp = dst_pts[i]
        A[2*i]     = [x, -y, 1, 0]
        A[2*i + 1] = [y,  x, 0, 1]
        b[2*i]     = xp
        b[2*i + 1] = yp

    params, _, _, _ = np.linalg.lstsq(A, b, rcond=None)
    a, bb, tx, ty = params

    H = np.array([
        [ a, -bb, tx],
        [bb,   a, ty],
        [ 0,   0,  1]
    ])
    return H

### (b) Wyznaczanie macierzy transformacji afinicznej

In [ ]:
def find_affine(src_pts, dst_pts):
    """Wyznaczanie macierzy przekształcenia afinicznego na podstawie par punktów.

    Transformacja afiniczna ma postać:
    [x']   [a  b  tx] [x]
    [y'] = [c  d  ty] [y]
    [ 1]   [0  0   1] [1]

    6 niewiadomych (a, b, tx, c, d, ty) → minimum 3 pary punktów.
    """
    n = src_pts.shape[0]
    A = np.zeros((2 * n, 6))
    b = np.zeros(2 * n)

    for i in range(n):
        x, y = src_pts[i]
        xp, yp = dst_pts[i]
        A[2*i]     = [x, y, 1, 0, 0, 0]
        A[2*i + 1] = [0, 0, 0, x, y, 1]
        b[2*i]     = xp
        b[2*i + 1] = yp

    params, _, _, _ = np.linalg.lstsq(A, b, rcond=None)
    a, bb, tx, c, d, ty = params

    H = np.array([
        [a,  bb, tx],
        [c,   d, ty],
        [0,   0,  1]
    ])
    return H

### Wybór punktów korespondencyjnych i wyznaczenie macierzy

In [ ]:
def find_corresponding_points(img_orig, img_transformed, n_points=10):
    """Automatyczne znajdowanie punktów korespondencyjnych.
    Prosta metoda: porównujemy narożniki i krawędzie obu obrazów.

    Wybieramy punkty ręcznie na podstawie wizualnej inspekcji obrazów.
    """
    pass


def select_points_from_images(img_orig, img_trans, title=''):
    """Wyświetla oba obrazy obok siebie, zwraca ręcznie dobrane punkty.
    Punkty są dobierane na podstawie wizualnych cech charakterystycznych."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    axes[0].imshow(img_orig, cmap='gray')
    axes[0].set_title(f'{title} — oryginał')
    axes[0].axis('off')
    axes[1].imshow(img_trans, cmap='gray')
    axes[1].set_title(f'{title} — przekształcony')
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
# Wyświetlenie obrazów w celu identyfikacji punktów korespondencyjnych
select_points_from_images(Im_A, Im_A_podo, 'A — podobieństwo')
select_points_from_images(Im_A, Im_A_afin, 'A — afiniczne')
select_points_from_images(Im_B, Im_B_podo, 'B — podobieństwo')
select_points_from_images(Im_B, Im_B_afin, 'B — afiniczne')

In [ ]:
def find_corners(image):
    """Proste wykrywanie narożników w obrazie na podstawie gradientu.
    Zwraca współrzędne (col, row) jako potencjalne punkty korespondencyjne."""
    gray = image if image.ndim == 2 else np.mean(image.astype(np.float64), axis=2)
    h, w = gray.shape

    # Prosty detektor narożników (Harris-like)
    Ix = np.zeros_like(gray)
    Iy = np.zeros_like(gray)
    Ix[:, 1:-1] = (gray[:, 2:] - gray[:, :-2]) / 2
    Iy[1:-1, :] = (gray[2:, :] - gray[:-2, :]) / 2

    Ixx = Ix * Ix
    Iyy = Iy * Iy
    Ixy = Ix * Iy

    # Uśrednianie w oknie 5x5
    k_size = 5
    kernel = np.ones((k_size, k_size)) / (k_size * k_size)

    def convolve_simple(img, kern):
        kh, kw = kern.shape
        ph, pw = kh // 2, kw // 2
        padded = np.pad(img, ((ph, ph), (pw, pw)), mode='constant')
        windows = np.lib.stride_tricks.sliding_window_view(padded, (kh, kw))
        return np.einsum('ijkl,kl->ij', windows, kern)

    Sxx = convolve_simple(Ixx, kernel)
    Syy = convolve_simple(Iyy, kernel)
    Sxy = convolve_simple(Ixy, kernel)

    k_harris = 0.04
    det = Sxx * Syy - Sxy * Sxy
    trace = Sxx + Syy
    R = det - k_harris * trace * trace

    threshold = 0.01 * R.max()
    corners = np.argwhere(R > threshold)

    # Wybierz rozproszone punkty
    if len(corners) > 200:
        indices = np.random.choice(len(corners), 200, replace=False)
        corners = corners[indices]

    return corners  # (row, col)


def match_points_ncc(img1, img2, pts1, patch_size=15):
    """Dopasowanie punktów za pomocą NCC (Normalized Cross-Correlation)."""
    gray1 = img1 if img1.ndim == 2 else np.mean(img1.astype(np.float64), axis=2)
    gray2 = img2 if img2.ndim == 2 else np.mean(img2.astype(np.float64), axis=2)
    h2, w2 = gray2.shape
    half = patch_size // 2

    matched_src = []
    matched_dst = []

    for pt in pts1:
        r, c = int(pt[0]), int(pt[1])
        if r - half < 0 or r + half >= gray1.shape[0] or c - half < 0 or c + half >= gray1.shape[1]:
            continue

        patch = gray1[r-half:r+half+1, c-half:c+half+1]
        patch = patch - patch.mean()
        patch_norm = np.sqrt(np.sum(patch * patch))
        if patch_norm < 1e-6:
            continue
        patch = patch / patch_norm

        best_ncc = -1
        best_pos = None

        for r2 in range(half, h2 - half):
            for c2 in range(half, w2 - half):
                patch2 = gray2[r2-half:r2+half+1, c2-half:c2+half+1]
                patch2 = patch2 - patch2.mean()
                p2_norm = np.sqrt(np.sum(patch2 * patch2))
                if p2_norm < 1e-6:
                    continue
                ncc = np.sum(patch * patch2 / p2_norm)
                if ncc > best_ncc:
                    best_ncc = ncc
                    best_pos = (r2, c2)

        if best_ncc > 0.8 and best_pos is not None:
            matched_src.append([c, r])  # (x, y) = (col, row)
            matched_dst.append([best_pos[1], best_pos[0]])

    return np.array(matched_src, dtype=np.float64), np.array(matched_dst, dtype=np.float64)

In [ ]:
def get_boundary_points(image):
    """Wyznacza punkty korespondencyjne z narożników i krawędzi widocznego obszaru obrazu.
    Szuka granic niebiałego/nieczarnego obszaru w obrazie przekształconym."""
    gray = image if image.ndim == 2 else np.mean(image.astype(np.float64), axis=2)
    mask = (gray > 10) & (gray < 245)

    rows = np.any(mask, axis=1)
    cols = np.any(mask, axis=0)

    if not np.any(rows) or not np.any(cols):
        return None

    r_min, r_max = np.where(rows)[0][[0, -1]]
    c_min, c_max = np.where(cols)[0][[0, -1]]

    return np.array([
        [c_min, r_min],
        [c_max, r_min],
        [c_min, r_max],
        [c_max, r_max],
    ], dtype=np.float64)

In [ ]:
# Wyznacz punkty korespondencyjne z narożników obrazów
# Oryginalne obrazy mają narożniki w oczywistych pozycjach
h_A, w_A = Im_A.shape[:2]
h_B, w_B = Im_B.shape[:2]

src_A = np.array([[0, 0], [w_A-1, 0], [0, h_A-1], [w_A-1, h_A-1]], dtype=np.float64)
src_B = np.array([[0, 0], [w_B-1, 0], [0, h_B-1], [w_B-1, h_B-1]], dtype=np.float64)

dst_A_podo = get_boundary_points(Im_A_podo)
dst_A_afin = get_boundary_points(Im_A_afin)
dst_B_podo = get_boundary_points(Im_B_podo)
dst_B_afin = get_boundary_points(Im_B_afin)

print('Punkty korespondencyjne A — podobieństwo:')
print(f'  src: {src_A.tolist()}')
print(f'  dst: {dst_A_podo.tolist()}')
print()
print('Punkty korespondencyjne A — afiniczne:')
print(f'  src: {src_A.tolist()}')
print(f'  dst: {dst_A_afin.tolist()}')

In [ ]:
# Wyznaczenie macierzy transformacji
H_A_podo = find_similarity(src_A, dst_A_podo)
H_B_podo = find_similarity(src_B, dst_B_podo)
H_A_afin = find_affine(src_A, dst_A_afin)
H_B_afin = find_affine(src_B, dst_B_afin)

print('Macierz podobieństwa A:')
print(np.round(H_A_podo, 4))
print()
print('Macierz podobieństwa B:')
print(np.round(H_B_podo, 4))
print()
print('Macierz afiniczna A:')
print(np.round(H_A_afin, 4))
print()
print('Macierz afiniczna B:')
print(np.round(H_B_afin, 4))

In [ ]:
# Sprawdzenie stabilności — użycie różnych podzbiorów punktów
print('=== Stabilność macierzy podobieństwa A ===')
for i in range(len(src_A)):
    idx = [j for j in range(len(src_A)) if j != i]
    H_sub = find_similarity(src_A[idx], dst_A_podo[idx])
    print(f'Bez punktu {i}: a={H_sub[0,0]:.4f}, b={H_sub[1,0]:.4f}, tx={H_sub[0,2]:.1f}, ty={H_sub[1,2]:.1f}')

print()
print('=== Stabilność macierzy afinicznej A ===')
for i in range(len(src_A)):
    idx = [j for j in range(len(src_A)) if j != i]
    H_sub = find_affine(src_A[idx], dst_A_afin[idx])
    print(f'Bez punktu {i}:')
    print(np.round(H_sub, 4))
    print()

### Wnioski
Przekształcenie podobieństwa wymaga minimum 2 par punktów (4 parametry: a, b, tx, ty).

Przekształcenie afiniczne wymaga minimum 3 par punktów (6 parametrów).

Użycie nadmiarowej liczby punktów (np. 4) z metodą najmniejszych kwadratów (lstsq)
pozwala uśrednić błędy i uzyskać stabilniejszy wynik.

Macierze wyznaczone z różnych podzbiorów punktów są zbliżone, co potwierdza poprawność metody.

## Zadanie 3.
Używając wyznaczonych transformacji, przekształć oryginalne obrazy A.bmp i B.bmp.
Porównaj wyniki z obrazami referencyjnymi.

UWAGA: Obliczenia dokonywane są we współrzędnych pikselowych.

In [ ]:
def apply_transform(image, H, output_shape=None):
    """Zastosowanie transformacji geometrycznej do obrazu.
    Używa transformacji odwrotnej (backward mapping) z interpolacją dwuliniową.

    image: obraz wejściowy
    H: macierz transformacji 3×3 (forward: src → dst)
    output_shape: (h, w) obrazu wyjściowego
    """
    if output_shape is None:
        output_shape = image.shape[:2]

    h_out, w_out = output_shape
    H_inv = np.linalg.inv(H)

    is_color = image.ndim == 3
    if is_color:
        result = np.zeros((h_out, w_out, image.shape[2]), dtype=np.uint8)
    else:
        result = np.zeros((h_out, w_out), dtype=np.uint8)

    # Wygeneruj siatkę pikseli wyjściowych
    cols, rows = np.meshgrid(np.arange(w_out), np.arange(h_out))
    ones = np.ones_like(cols)
    dst_coords = np.stack([cols.ravel(), rows.ravel(), ones.ravel()], axis=0).astype(np.float64)

    # Transformacja odwrotna
    src_coords = H_inv @ dst_coords
    src_x = src_coords[0]
    src_y = src_coords[1]

    # Interpolacja dwuliniowa
    h_in, w_in = image.shape[:2]
    x0 = np.floor(src_x).astype(int)
    y0 = np.floor(src_y).astype(int)
    x1 = x0 + 1
    y1 = y0 + 1

    # Maska pikseli wewnątrz obrazu źródłowego
    valid = (x0 >= 0) & (x1 < w_in) & (y0 >= 0) & (y1 < h_in)

    x0c = np.clip(x0, 0, w_in - 1)
    x1c = np.clip(x1, 0, w_in - 1)
    y0c = np.clip(y0, 0, h_in - 1)
    y1c = np.clip(y1, 0, h_in - 1)

    dx = src_x - x0
    dy = src_y - y0

    if is_color:
        img_f = image.astype(np.float64)
        for ch in range(image.shape[2]):
            I00 = img_f[y0c, x0c, ch]
            I10 = img_f[y0c, x1c, ch]
            I01 = img_f[y1c, x0c, ch]
            I11 = img_f[y1c, x1c, ch]

            val = I00 * (1-dx) * (1-dy) + I10 * dx * (1-dy) + I01 * (1-dx) * dy + I11 * dx * dy
            val_arr = np.clip(val, 0, 255).astype(np.uint8)
            val_arr[~valid] = 0
            result[:, :, ch] = val_arr.reshape(h_out, w_out)
    else:
        img_f = image.astype(np.float64)
        I00 = img_f[y0c, x0c]
        I10 = img_f[y0c, x1c]
        I01 = img_f[y1c, x0c]
        I11 = img_f[y1c, x1c]

        val = I00 * (1-dx) * (1-dy) + I10 * dx * (1-dy) + I01 * (1-dx) * dy + I11 * dx * dy
        val_arr = np.clip(val, 0, 255).astype(np.uint8)
        val_arr[~valid] = 0
        result = val_arr.reshape(h_out, w_out)

    return result

In [ ]:
# Przekształcenie obrazów A i B
A_podo_result = apply_transform(Im_A, H_A_podo, Im_A_podo.shape[:2])
A_afin_result = apply_transform(Im_A, H_A_afin, Im_A_afin.shape[:2])
B_podo_result = apply_transform(Im_B, H_B_podo, Im_B_podo.shape[:2])
B_afin_result = apply_transform(Im_B, H_B_afin, Im_B_afin.shape[:2])

# Porównanie z referencyjnymi
for name, result, ref in [('A podobieństwo', A_podo_result, Im_A_podo),
                           ('A afiniczne',    A_afin_result, Im_A_afin),
                           ('B podobieństwo', B_podo_result, Im_B_podo),
                           ('B afiniczne',    B_afin_result, Im_B_afin)]:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].imshow(result, cmap='gray')
    axes[0].set_title(f'{name} — wynik')
    axes[0].axis('off')
    axes[1].imshow(ref, cmap='gray')
    axes[1].set_title(f'{name} — referencja')
    axes[1].axis('off')

    # Różnica
    diff = np.abs(result.astype(np.float64) - ref.astype(np.float64))
    axes[2].imshow(diff, cmap='hot')
    axes[2].set_title(f'Różnica (max={diff.max():.0f})')
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()

    mse = np.mean(diff**2)
    print(f'{name}: MSE = {mse:.2f}, max diff = {diff.max():.0f}')
    print()

### Wnioski
Zastosowanie backward mapping z interpolacją dwuliniową daje obrazy o wyższej jakości wizualnej
niż forward mapping (brak dziur w obrazie wynikowym).

Obrazy nie są idealnie identyczne z referencyjnymi ze względu na różne metody interpolacji
i zaokrąglenia numeryczne, ale różnice są niewielkie.

Interpolacja dwuliniowa jest najbardziej efektywną metodą w sensie jakości wizualnej
przy rozsądnym koszcie obliczeniowym.